# ShadowWork-SHIFT | Publish reproducible SW12 + SW17 models and datasets to Hugging Face

**Purpose:** CPU-only publication of the actual, previously trained SW12 v3 and SW17 v4 adapters, exact 800-row-per-arm training JSONLs, precommitted split files, construction audits and evaluation traces. No training or GPU is required.

**Required Kaggle inputs:** `shaurya4667/shadowwork-12-closed-loop`; completed SW17 output folder either preserved at `/kaggle/working/shadowwork/17_training_attribution` in this session or attached as a Kaggle dataset containing extracted SW17 output files; a write-enabled Hugging Face `HF_TOKEN` in Kaggle Secrets.

**Integrity:** refuses to proceed when SW17 scoring or sealed episode collection is incomplete. Adapter files and exact training files are preserved; changing `adapter_config.json` to reference the verified public custom base model is metadata only, with the original config retained.

**Credible methodology:** The released [dataset card](../docs/HF_DATASET_CARD.md) and [construction protocol](../docs/DATASET_CONSTRUCTION.md) disclose programmatically generated, SQLite-executed synthetic tasks, learner-visited corrections, scripted reference labeling, paired controls, and the absence of independent human annotations or external evaluation.


In [ ]:
%pip install -q "huggingface_hub>=0.34,<2"

## 1. Editable settings and authentication

In [ ]:
from pathlib import Path
import json
import hashlib
import shutil
import zipfile
from huggingface_hub import HfApi, hf_hub_download
from kaggle_secrets import UserSecretsClient

OWNER = "i-am-shaurya05"
BASE_MODEL_ID = f"{OWNER}/shadowwork-shift-plain-sft"
DATASET_REPO_ID = f"{OWNER}/shadowwork-shift-datasets"
MODEL_COLLECTION_SLUG = f"{OWNER}/shadowwork-shift-models-and-frozen-adapters-6af5c8a5a548fb703ab420ac"
DATA_COLLECTION_SLUG = f"{OWNER}/shadowwork-shift-training-data-and-audit-evidence-6af5c8a51d44c02ee6e19812"

SW12_ROOT = Path("/kaggle/input/datasets/shaurya4667/shadowwork-12-closed-loop")
SW17_ROOT = Path("/kaggle/working/shadowwork/17_training_attribution")
RESULTS_ROOT = Path("/kaggle/input")
STAGING_ROOT = Path("/kaggle/working/shift_publication")
STAGING_ROOT.mkdir(parents=True, exist_ok=True)

# A new Kaggle notebook/session needs the completed SW17 output attached
# as a Kaggle input dataset. Never regenerate or train SW17 here.
if not (SW17_ROOT / "learner_corrective_v4_training.jsonl").is_file():
    candidates = sorted(
        p.parent for p in RESULTS_ROOT.rglob("learner_corrective_v4_training.jsonl")
    )
    assert len(candidates) == 1, (
        "Attach the already-completed SW17 output as a Kaggle dataset, "
        "or run this publisher in the original SW17 session. Candidates:", candidates
    )
    SW17_ROOT = candidates[0]

assert SW12_ROOT.is_dir(), f"Attach SW12 dataset: {SW12_ROOT}"
assert (SW17_ROOT / "final_provenance.json").is_file(), (
    "Run the final SW17 evaluation, scoring, and export cells before publication."
)
assert (SW17_ROOT / "learning_gate.json").is_file()
assert (SW17_ROOT / "policy_summary.csv").is_file()
assert (SW17_ROOT / "precommit.json").is_file()
assert (SW17_ROOT / "selection_audit.json").is_file()
for arm in ("rehearsal_v4", "learner_corrective_v4"):
    assert (SW17_ROOT / f"{arm}_training.jsonl").is_file()
    assert (SW17_ROOT / f"{arm}_training_metadata.json").is_file()
    assert (SW17_ROOT / f"{arm}_adapter" / "adapter_model.safetensors").is_file()

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
assert HF_TOKEN, "Add a write-enabled HF_TOKEN in Kaggle Secrets"
api = HfApi(token=HF_TOKEN)
identity = api.whoami()
assert identity.get("name") == OWNER, ("Token owner mismatch", identity.get("name"))
base = api.model_info(BASE_MODEL_ID)
assert not base.private and not base.gated, "Exact custom full base must be public and ungated"
print("Publishing as:", OWNER)
print("SW12 source:", SW12_ROOT)
print("SW17 completed source:", SW17_ROOT)
print("Exact public base:", BASE_MODEL_ID)


## 2. Verify original adapter files, publish exact weights with corrected public base linkage

In [ ]:
# Both v3 and v4 adapters are *already trained*. Publication never calls an optimizer.
ADAPTERS = {
    "learner_corrective_v3": (f"{OWNER}/shadowwork-shift-corrective-v3",
                              SW12_ROOT / "learner_corrective_v3_adapter", "SW12"),
    "rehearsal_v3": (f"{OWNER}/shadowwork-shift-rehearsal-v3",
                     SW12_ROOT / "rehearsal_v3_adapter", "SW12"),
    "learner_corrective_v4": (f"{OWNER}/shadowwork-shift-corrective-v4",
                              SW17_ROOT / "learner_corrective_v4_adapter", "SW17"),
    "rehearsal_v4": (f"{OWNER}/shadowwork-shift-rehearsal-v4",
                     SW17_ROOT / "rehearsal_v4_adapter", "SW17"),
}

for arm, (repo_id, source, experiment) in ADAPTERS.items():
    assert source.is_dir() and (source / "adapter_model.safetensors").is_file()
    assert (source / "adapter_config.json").is_file()
    original_config = json.loads((source / "adapter_config.json").read_text())
    original_base = original_config.get("base_model_name_or_path", "")
    assert original_base == BASE_MODEL_ID or "shadowork_123" in original_base, (
        "Unexpected pretrained base reference", arm, original_base
    )

    def filehash(path):
        h = hashlib.sha256()
        with path.open("rb") as f:
            for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
                h.update(chunk)
        return h.hexdigest()

    original_sha = filehash(source / "adapter_model.safetensors")
    model_dir = STAGING_ROOT / "model_releases" / arm
    if model_dir.exists():
        shutil.rmtree(model_dir)
    model_dir.mkdir(parents=True)
    for name in (
        "adapter_model.safetensors", "adapter_config.json", "tokenizer.json",
        "tokenizer_config.json", "chat_template.jinja", "special_tokens_map.json",
    ):
        if (source / name).is_file():
            shutil.copy2(source / name, model_dir / name)

    public_config = dict(original_config)
    public_config["base_model_name_or_path"] = BASE_MODEL_ID
    (model_dir / "adapter_config.json").write_text(json.dumps(public_config, indent=2) + "\n")
    provenance = model_dir / "provenance"
    provenance.mkdir()
    (provenance / "original_adapter_config.json").write_text(json.dumps(original_config, indent=2) + "\n")
    (provenance / "original_weights_sha256.txt").write_text(original_sha + "\n")

    desc = ("learner-visited reference corrections" if arm.startswith("learner_")
            else "ordinary expert-trajectory rehearsal")
    model_card = f"""---
library_name: peft
pipeline_tag: text-generation
license: cc-by-nc-4.0
base_model: {BASE_MODEL_ID}
base_model_relation: adapter
tags:
  - lexsi
  - agenttune
  - auditkit
  - shadowwork-shift
  - lora
---

# ShadowWork-SHIFT {arm} ({experiment})

This is the **actual previously trained LoRA adapter**, optimized on {desc}.
Both arms in {experiment} start from the same original SW09 corrective adapter and use the same
800-row training budget, 80 optimizer steps, learning rate 5e-5 and global batch size 8.

**Exact base model:** https://huggingface.co/{BASE_MODEL_ID}
**Exact training rows, splits, method, safety and replay evidence:** https://huggingface.co/datasets/{DATASET_REPO_ID}
**Original experiment source:** https://github.com/githubshaurya/ShadowWork-SHIFT
**Training method:** synthetic SQLite tasks, model-visited corrective states, scripted reference
action labels, matched rehearsal controls, precommitted split SHA-256 and fixed-controller evaluation.
The reference labels are programmatic and **not human expert annotations**.

The weight file bytes are unchanged. The public base checkpoint reference in adapter_config.json
replaces a local Kaggle path **only as metadata**; original weights SHA-256 and configuration
are retained under provenance/.

SW12 sealed 64-task baseline (no unified controller): frozen SW09 24/64;
corrective v3 46/64; rehearsal v3 47/64. SW17 outcome numbers must be read from its
released policy_summary.csv; training loss is not a task-success metric.
These evaluations are synthetic and do not demonstrate independent external generalization.

License: CC BY-NC 4.0, with applicable upstream restrictions.
"""
    (model_dir / "README.md").write_text(model_card)
    api.create_repo(repo_id, repo_type="model", private=False, exist_ok=True)
    api.upload_folder(
        folder_path=str(model_dir), repo_id=repo_id, repo_type="model",
        commit_message=f"Publish original {arm} adapter, source SHA and model card"
    )
    public = api.model_info(repo_id)
    names = {item.rfilename for item in public.siblings}
    assert not public.private and not public.gated
    assert {"adapter_model.safetensors", "adapter_config.json", "README.md"} <= names
    remote_config = hf_hub_download(
        repo_id, "adapter_config.json", repo_type="model",
        token=HF_TOKEN, force_download=True
    )
    assert json.loads(Path(remote_config).read_text())["base_model_name_or_path"] == BASE_MODEL_ID
    print("Verified public model:", f"https://huggingface.co/{repo_id}", original_sha)


## 3. Publish SW12 training metadata, datasets, exact scored episodes, and optional SW13–SW16 evidence

In [ ]:
# Dataset release: exact rows, generator split commitments, learner rollouts,
# audited episode traces, scoring and construction methodology.
api.create_repo(DATASET_REPO_ID, repo_type="dataset", private=False, exist_ok=True)
records = STAGING_ROOT / "dataset_release"
if records.exists():
    shutil.rmtree(records)
records.mkdir(parents=True)

def copy_source(source_root, name, prefix):
    file = source_root / name
    assert file.is_file(), f"Required evidence missing: {file}"
    target = records / prefix / name
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(file, target)

sw12_files = [
    "all_episode_scorecard.csv", "final_provenance.json", "grouped_summary.csv",
    "precommit.json", "selection_audit.json", "train_tasks.jsonl",
    "dev_tasks.jsonl", "sealed_tasks.jsonl",
    "learner_corrective_v3_training.jsonl", "rehearsal_v3_training.jsonl",
    "learner_corrective_v3_training_metadata.json", "rehearsal_v3_training_metadata.json",
    "learner_corrective_v3_tokenized_source_sha256.txt",
    "rehearsal_v3_tokenized_source_sha256.txt",
]
for name in sw12_files:
    copy_source(SW12_ROOT, name, "sw12")

sw12_episodes = sorted((SW12_ROOT / "evaluation").rglob("*.json"))
assert len(sw12_episodes) >= 200, "SW12 evaluation traces are missing"
for source in sw12_episodes:
    dest = records / "sw12" / "evaluation" / source.relative_to(SW12_ROOT / "evaluation")
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, dest)

sw17_files = [
    "precommit.json", "final_provenance.json", "selection_audit.json",
    "train_tasks.jsonl", "dev_tasks.jsonl", "sealed_tasks.jsonl",
    "rehearsal_v4_training.jsonl", "learner_corrective_v4_training.jsonl",
    "rehearsal_v4_training_metadata.json", "learner_corrective_v4_training_metadata.json",
    "episode_scorecard.csv", "policy_summary.csv", "group_summary.csv",
    "unseen_schema_summary.csv", "paired_tests.csv", "learning_gate.json",
]
for name in sw17_files:
    copy_source(SW17_ROOT, name, "sw17")

with (SW17_ROOT / "precommit.json").open() as f:
    precommit = json.load(f)
assert precommit["experiment"] == "SW17-learner-repair-vs-matched-expert"
assert precommit["seed"] == 20261115
for split, expected in precommit["split_sha256"].items():
    file = SW17_ROOT / f"{split}_tasks.jsonl"
    digest = hashlib.sha256(file.read_bytes()).hexdigest()
    assert digest == expected, ("Frozen SW17 split changed", split)

for arm in ("rehearsal_v4", "learner_corrective_v4"):
    metadata = json.loads((SW17_ROOT / f"{arm}_training_metadata.json").read_text())
    assert metadata["arm"] == arm and metadata["steps"] == 80
    assert metadata["train_rows"] == 800 and metadata["global_batch"] == 8
    actual = hashlib.sha256((SW17_ROOT / f"{arm}_training.jsonl").read_bytes()).hexdigest()
    assert actual == metadata["training_sha256"], ("SW17 training rows changed", arm)

for policy in ("frozen_sw09", "corrective_v4", "rehearsal_v4"):
    traces = sorted((SW17_ROOT / "evaluation" / "sealed" / policy).glob("*.json"))
    assert len(traces) == 80, ("Incomplete SW17 sealed evaluation", policy, len(traces))

sw17_collection = sorted((SW17_ROOT / "collection").glob("*.json"))
assert len(sw17_collection) == 160, "Missing original train-only learner rollouts"
sw17_episodes = sorted((SW17_ROOT / "evaluation").rglob("*.json"))
assert len(sw17_episodes) >= 240, "Missing evaluated SW17 episodes"

for source in sw17_collection:
    dest = records / "sw17" / "collection" / source.name
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, dest)
for source in sw17_episodes:
    dest = records / "sw17" / "evaluation" / source.relative_to(SW17_ROOT / "evaluation")
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, dest)

# Older SW13-SW16 source archives are optional because their Kaggle attachments
# may be separate. Never describe absent optional archives as uploaded.
optional_names = (
    "ShadowWork_13_Live_Results.zip",
    "ShadowWork_14_Controller_Repair_Results1.zip",
    "ShadowWork_15_Cross_Protocol_Recovery_Results.zip",
    "ShadowWork_16_Unified_Recovery_Results.zip",
    "ShadowWork_16_Budget_Regression_Results.zip",
)
for name in optional_names:
    available = sorted(RESULTS_ROOT.rglob(name))
    if not available:
        print("Optional archive not attached:", name)
        continue
    with zipfile.ZipFile(available[0]) as z:
        assert z.testzip() is None, ("Corrupt archive", available[0])
    dest = records / "archived_evidence" / "source_archives" / name
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(available[0], dest)

# Source-reviewed dataset card and transparent construction methodology,
# embedded to avoid additional external downloads during publishing.
dataset_card = "---\nlicense: cc-by-nc-4.0\nlanguage:\n  - en\npretty_name: ShadowWork-SHIFT Self-Healing Tool-Agent Training and Audited Traces\ntags:\n  - agents\n  - tool-use\n  - synthetic\n  - evaluation\n  - agenttune\n  - auditkit\n  - self-healing\n  - lexsi\n---\n\n# ShadowWork-SHIFT | Synthetic self-healing agent dataset\n\n**Provenance:** Deterministically generated synthetic stateful-maintenance tasks; actual learner rollouts on train-only cases; observation-conditioned, scripted reference continuations; matched rehearsal controls; executable SQLite replay and SHA-256 split/row commitments. **No human-expert annotation, production incident collection or independent external benchmark is claimed.**\n\n**Data publication target:** `i-am-shaurya05/shadowwork-shift-datasets`. See the original SW09 source data in [shadowwork-shift-evidence-and-training](https://huggingface.co/datasets/i-am-shaurya05/shadowwork-shift-evidence-and-training).\n\n## Why the dataset was created\n\nStateful tool agents can receive successful write receipts without persisted changes, observe stale reads, miss mandatory verification checks, or attempt unauthorized mutations. These records support controlled before/after studies of whether training on *states the agent actually visits* improves its one-action decisions beyond ordinary expert demonstration rehearsal, under the same recovery controller.\n\n## How the data was created\n\n1. A scripted task generator samples field/value changes, permission and approval states, health gates, verification protocols, and predefined fault schedules.\n2. Tasks are split by ID into train, development and sealed test cohorts using fixed random seeds; exact JSONL split hashes are committed before evaluation.\n3. A rule-based reference policy uses observed ticket/runbook/health/approval/lease/tool outputs, proposes the next tool action, and is tested by executing each reference trajectory against persisted SQLite state.\n4. The **frozen original SW09 corrective agent** is run on train-only tasks; its actual action/result histories are recorded without access to test labels.\n5. Each learner-visited prompt receives a reference-policy next-action label. Proposed-action/reference disagreement is logged and used to prioritize actual corrections.\n6. Two **matched 800-row datasets** are built per experiment. Each arm shares 400 expert-anchor rows; the remaining 400 are learner-visited corrective examples or ordinary expert-state rehearsal examples. SW17 additionally matches rehearsal examples on fault category, one-/two-check protocol and action class when available.\n7. Tool histories are replayed and checked for consistency, examples are token-budget-filtered, and finalized JSONL files are hashed. Neither the script nor the release claims human annotation.\n\n## Dataset composition\n\n| Content | SW12 | SW17 |\n|---|---:|---:|\n| Training tasks | 192 | 240 |\n| Development tasks | 32 | 40 |\n| Sealed task instances | 64 | 80 |\n| Frozen learner rollouts from train-only tasks | 160 | 160 |\n| Training examples per curriculum | 800 | 800 |\n| Shared expert anchors per curriculum | 400 | 400 |\n| Specialized examples per curriculum | 400 | 400 |\n\nSW12 contains eight fault families (`normal`, `double_conflict`, `double_noop`, `wrong_then_stale`, `two_stale`, `false_positive`, `injected_forbidden`, `injected_authorized`). SW17 adds `approval_not_required` and `lease_expired`, plus unseen field names and changed service/region wording in the sealed split. One- and two-check protocols are both represented.\n\n## File guide\n\n- `sw12/train_tasks.jsonl`, `dev_tasks.jsonl`, `sealed_tasks.jsonl`: synthetic task parameters and original split identities.\n- `sw12/learner_corrective_v3_training.jsonl`, `rehearsal_v3_training.jsonl`: exact matched v3 training rows.\n- `sw12/selection_audit.json`, `precommit.json`, `*_training_metadata.json`: data construction counts, hashes and frozen settings.\n- `sw12/evaluation/`: recorded, task-level evaluation runs with actions, observations and scores.\n- `sw17/train_tasks.jsonl`, `dev_tasks.jsonl`, `sealed_tasks.jsonl`: the later, separate shifted task generator and frozen split.\n- `sw17/learner_corrective_v4_training.jsonl`, `rehearsal_v4_training.jsonl`: exact matched v4 training rows.\n- `sw17/selection_audit.json`, `precommit.json`, `*_training_metadata.json`: training provenance.\n- `sw17/collection/` and `sw17/evaluation/`: train-only learner rollouts and sealed evaluation traces.\n- `archived_evidence/`: selected SW09/SW13–SW16 original archives when supplied at publishing time.\n\n**Availability note:** only files listed in the repository's **Files and versions** tab should be considered published. The file guide is a release schema, not a claim that every optional archived file was already uploaded.\n\n## Training and evaluation\n\nBoth v3 and v4 comparisons use the **same frozen SW09 corrective LoRA starting point**, matched optimizer budget (80 steps, 800 rows/arm, effective batch 8, learning rate `5e-5`), and a shared controller **within each comparison**. The model base is a specific, public Tiny Aya-derived full checkpoint; it is not interchangeable with upstream Tiny Aya weights.\n\n**Previously established SW12 sealed 64-task result before the final unified controller:** frozen original 24/64, corrective v3 46/64, rehearsal v3 47/64 strict successes. This demonstrates model-adaptation gains but not corrective-curriculum superiority. SW17 results must be reported separately after their complete scored evaluation, not filled in from training loss.\n\n## Quality checks\n\nAll reference tasks must terminate with correct persisted-state/evidence outcomes within the episode budget. Training-only learner histories are replayed against the same simulator; candidate action/reference mismatches, coverage, token-budget filtering, deduplication and matched-row counts are logged. Evaluation uses action/result traces, SQLite state, AuditKIT recorded checks and paired strict-success statistics. Original split files and training JSONLs have SHA-256 integrity commitments.\n\n**Limitations:** fully synthetic, author-scripted reference labels, two related protocol environments, limited adversarial and external coverage. These data do **not** establish safe deployment or real-world recovery efficacy. A shared simulator's successful reference policy is not proof of independent correctness, and 800 action rows are not 800 independent tasks.\n\n## Use and license\n\nFor **non-commercial research, hackathon evaluation and reproducible controlled tool-agent recovery studies** under CC BY-NC 4.0 and the applicable upstream model/tool terms. Do not treat any simulated results as operational safety guarantees.\n\n**Complete, auditable methodology and source:** [GitHub dataset construction and provenance](https://github.com/githubshaurya/ShadowWork-SHIFT/blob/main/docs/DATASET_CONSTRUCTION.md), [SW12 notebook](https://github.com/githubshaurya/ShadowWork-SHIFT/blob/main/notebooks/SW12_Closed_Loop_Adaptation.ipynb), [SW17 notebook](https://github.com/githubshaurya/ShadowWork-SHIFT/blob/main/notebooks/SW17_Fixed_Controller_Learning_Attribution_Kaggle.ipynb).\n\nDocumentation design follows the transparency prompts in Gebru et al., *Datasheets for Datasets* (2021), and [Hugging Face dataset card guidance](https://huggingface.co/docs/hub/datasets-cards). This does not imply external review or endorsement.\n"
construction_methodology = "# ShadowWork-SHIFT | Dataset construction and provenance\n\n**Scope:** LEXSI Track 3, Problem 8, self-healing tool agents. **Dataset type:** entirely synthetic, English-language, stateful configuration-maintenance workflows with executable SQLite state and fault injection. **Origin:** experiment-specific generators, frozen learner rollouts, and scripted reference continuations, not scraped production tickets and not independently human-annotated data.\n\n## 1. Motivation and what constitutes an example\n\nThe training examples teach one next tool action given a maintenance request, the four initial tool observations, and the sequence of tool calls/results. The objective is to train an agent to detect unsuccessful or unsafe state transitions, recover without unsafe mutation, and stop only after sufficient read-back evidence. Distinct fault classes include transient write conflicts, silently ignored writes, stale state reads, false-positive commit acknowledgments, wrong writes, injection-like untrusted notes, authorization denial, approval-policy edge cases, and expired write leases.\n\nA **training row** is `{\"prompt\": \"...\", \"completion\": \"[{...}]\"}`. It is a **single action-label pair**, not a standalone full human dialogue or an independent task. Many rows can originate from the same generated task or rollout. Each **task** has a request, initial value, intended value, authorization conditions, required verification count, and deterministic fault schedule. A **rollout** is an actual frozen model's tool-action trajectory executed in the simulator. A **label** is the next action selected by an executable scripted reference continuation.\n\nThe original SW09 data and 70-task final holdout are already in the public [SW09 dataset](https://huggingface.co/datasets/i-am-shaurya05/shadowwork-shift-evidence-and-training). SW12 and SW17 are newer, related experiments; do not combine their test cohorts into a single independent benchmark.\n\n## 2. Task design and provenance\n\n| Contract | SW12 | SW17 |\n|---|---:|---:|\n| Generator seed | `20261113` | `20261115` |\n| Train tasks | 192 (8 groups × 24) | 240 (10 groups × 24) |\n| Development tasks | 32 (8 × 4) | 40 (10 × 4) |\n| Sealed test tasks | 64 (8 × 8) | 80 (10 × 8) |\n| Train-only learner rollouts | 160 (20 per group) | 160 (16 per group) |\n| Per-arm supervised examples | 800 | 800 |\n| Common expert examples per arm | 400 | 400 |\n| Curriculum-specific examples per arm | 400 | 400 |\n| Max tool actions | 22 | 24 |\n| Verification protocols | One and two post-write checks | One and two post-write checks |\n| Shared starting policy | Original SW09 corrective LoRA | Original SW09 corrective LoRA |\n\nSW12 groups are `normal`, `double_conflict`, `double_noop`, `wrong_then_stale`, `two_stale`, `false_positive`, `injected_forbidden`, and `injected_authorized`. SW17 adds `approval_not_required` and `lease_expired`. Both model protocol variants within every group. Fault behavior is implemented by an executable `RecoveryEnvironment` that stores persisted configuration values in SQLite and returns tool results that can differ from durable state.\n\nSW17 further changes service names, region names, request phrasing, and includes three previously unseen configuration fields in part of the sealed test cohort. This is **shift within a related synthetic workflow family**, not generalization to independently authored real production systems.\n\n## 3. Generation pipeline\n\n1. **Programmatic scenario generation.** Construct tasks from explicit value domains, typed fields, authorization and approval states, a bounded canary-error policy, failure groups and one-/two-check protocol flags. Generate separate train/development/sealed identities using fixed seeds. The outputs `train_tasks.jsonl`, `dev_tasks.jsonl`, `sealed_tasks.jsonl`, `precommit.json` and their SHA-256 values are recorded before model evaluation.\n2. **Executable reference policy.** A rule-based next-action policy consumes structured tool observations such as ticket, runbook, health, approval, lease and committed-state checks, rather than using a separate language model judge. Its continuation is run against the SQLite environment and must achieve the defined strict-success condition on every generated task in the reference-solvability check. This is a **scripted reference**, not an externally certified optimal policy.\n3. **Frozen learner-state collection.** Run the fixed original SW09 corrective model on **training-only** task IDs. Save the learner's actions, tool outputs, scores and identity hashes per episode. SW12 and SW17 each stratify 160 rollouts by group before sampling.\n4. **Corrective label construction.** Replay each saved learner action through the same environment. At learner-visited states after the four scripted initial reads, ask the reference policy for the next action. Mark `correction=true` when the learner's original proposal differs from the reference. The teacher targets are **model-output-conditioned, programmatically validated labels**, not human ratings.\n5. **Length and coverage filtering.** Remove examples exceeding the SFT token budget; select a mixture across fault group, verification count and next-action category, giving priority to actual learner/reference mismatches. Confirm coverage of all relevant groups and both verification protocols; verify prompt uniqueness for selected corrective rows.\n6. **Matched counterfactual training sets.** Both arms contain 400 identical reference-policy rows. In the remaining 400, the corrective arm uses learner-visited states labeled by the reference. Rehearsal uses ordinary reference-policy states that do not require a learner mistake. SW12's rehearsal extras are the next 400 shuffled expert rows; SW17 more carefully tries to match each selected corrective row by group, protocol and next-action category, falling back to coarser matches when needed. The actual exact-matching count is recorded in SW17's `selection_audit.json`.\n7. **Frozen, equal-budget optimization.** Each arm starts from the exact same SW09 corrective adapter on the same custom Tiny Aya-derived full checkpoint. Both use 800 rows, 80 optimizer steps, learning rate `5e-5`, effective global batch size 8, and the same frozen per-experiment evaluation conditions. SW17's T4 training implementation uses explicit completion-only labels, 4-bit frozen base/FP16 compute and FP32 LoRA parameters; its earlier software-workaround history does not change the experimental intent.\n\n## 4. Verification and leakage boundaries\n\n- **Persistent state, not acknowledgment alone:** completion requires authorized final values in SQLite plus required post-write matching checks; an apparent successful commit can be a silent no-op or false positive.\n- **Forbidden actions:** authorization and argument-schema gates reject disallowed operations, and attempted unsafe mutations are logged separately from executed mutations.\n- **Reference replay:** generated tasks must be solvable by the scripted reference under the action budget; learner rollout histories are replayed and checked against their originally saved tool results.\n- **Sealed split commitment:** split SHA-256 digests and the `precommit.json` are saved before evaluation; existing files are compared and rejected if changed rather than silently regenerated.\n- **No training on sealed rows:** training construction references only `train_tasks`, training-only learner rollouts, and the training-split reference states; development and sealed IDs are disjoint.\n- **Paired scoring:** compare models on identical task IDs, a common fixed controller and common simulator. Report strict task success, persisted-state correctness, authorization proposals, model calls, token counts, and paired win/loss statistics where recorded.\n- **Source file hashes:** each arm's `*_training_metadata.json` records the SHA-256 of its exact JSONL training file. `selection_audit.json` records selected correction counts, distributions and hashes; `final_provenance.json` and episode-level JSON permit follow-up audit.\n\nThe checks above are **code-level controls**, not proof that the scripted teacher is optimal or that the synthetic task families fully represent production reality. They are also not independent assessments by outside annotators.\n\n## 5. Published fields and interpretation\n\n**Task JSONL** includes `id`, `split`, `group`, `key`, `old`, `target`, `service`, `region`, `request`, `authorized`, `approved`, `requires_approval`, `canary_errors`, `max_canary_errors`, `allowed`, `checks_required`.\n\n**Training JSONL** has `prompt` and `completion`. The prompt contains the tool-use rules, task and available action/result history; the completion is a JSON array containing exactly one structured tool action.\n\n**Collection episode JSON** preserves frozen-model histories, generated action strings, source adapter hash and outcome. **Evaluation episode JSON** preserves task IDs, group, verification protocol, executed actions and tool results, final stored values, strict score, AuditKIT verdict and resource-use counters.\n\n**Do not confuse:** 800 rows with 800 independent tasks; successful scripted reference rollout with independent ground-truth certification; simulator read-back with a real production database; or multiple model-task evaluations with that many distinct benchmark tasks.\n\n## 6. Licenses, suitability and limitations\n\nThe scenarios are synthetic, not derived from identifiable human subjects or confidential production logs. The public datasets are intended for controlled research/demos on stateful tool-use, failure detection and recovery. They do not contain evidence of safe performance on real business infrastructure and should not be used as a standalone safety certification for tool agents. The creator's chosen bundle license is **CC BY-NC 4.0**, and users must additionally observe applicable model and software licenses. Synthetic status does not eliminate the possibility of methodological bias.\n\n**Main limitations:** only two closely related simulator families; hand-scripted fault distributions and reference policy; limited independently authored adversarial conditions; no external human labeling or adjudication; limited held-out tasks per group; possible lexical correlations; repeated test-family iteration across notebooks; protocol assumptions embedded in the synthetic environment.\n\n## 7. Reproduce and audit\n\nSource notebooks:\n- [SW12 closed-loop dataset/training/evaluation](../notebooks/SW12_Closed_Loop_Adaptation.ipynb), especially the task generator and matched-data construction cells.\n- [SW17 fixed-controller learning attribution](../notebooks/SW17_Fixed_Controller_Learning_Attribution_Kaggle.ipynb), especially the frozen task split, real learner rollouts, and training-row selection cells.\n- [SW17 one-cell frozen evaluation](../notebooks/SW17_SINGLE_CELL_EVALUATION_AFTER_RESTART.ipynb).\n\nPublic dataset locations:\n- SW09 existing dataset: https://huggingface.co/datasets/i-am-shaurya05/shadowwork-shift-evidence-and-training\n- SW12–SW17 target dataset: https://huggingface.co/datasets/i-am-shaurya05/shadowwork-shift-datasets\n\nThe second link is a **publication target** until its files are actually uploaded and verified. The GitHub source documents this workflow, but doesn't by itself constitute a Hugging Face dataset release.\n\n## 8. Documentation conventions\n\nThis dataset is documented using the motivation, composition, collection process, preprocessing, intended use and limitation structure advocated by **Gebru et al., _Datasheets for Datasets_ (2021)** and Hugging Face's official [dataset card guidance](https://huggingface.co/docs/hub/datasets-cards). This is a documentation framework, **not** an endorsement or independent validation of our dataset by those authors.\n"
(records / "README.md").write_text(dataset_card)
(records / "documentation").mkdir(exist_ok=True)
(records / "documentation" / "DATASET_CONSTRUCTION.md").write_text(construction_methodology)

# Record SHA-256 for every released file, excluding the manifest itself.
def sha256_file(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

manifest = {
    file.relative_to(records).as_posix(): sha256_file(file)
    for file in sorted(records.rglob("*"))
    if file.is_file() and file.name != "release_manifest_sha256.json"
}
(records / "release_manifest_sha256.json").write_text(
    json.dumps(manifest, indent=2, sort_keys=True) + "\n"
)
print("Files ready for public dataset:", len(manifest))
print("SW12 scored episodes:", len(sw12_episodes))
print("SW17 collection rollouts:", len(sw17_collection))
print("SW17 scored episodes:", len(sw17_episodes))

api.upload_folder(
    folder_path=str(records), repo_id=DATASET_REPO_ID, repo_type="dataset",
    commit_message="Release audited SW12 and SW17 matched training data, trace evidence and dataset card",
)
public = api.dataset_info(DATASET_REPO_ID)
public_files = {item.rfilename for item in public.siblings}
assert not public.private
assert {
    "README.md", "documentation/DATASET_CONSTRUCTION.md", "release_manifest_sha256.json",
    "sw12/learner_corrective_v3_training.jsonl", "sw12/rehearsal_v3_training.jsonl",
    "sw17/learner_corrective_v4_training.jsonl", "sw17/rehearsal_v4_training.jsonl",
    "sw17/paired_tests.csv", "sw17/selection_audit.json",
} <= public_files, ("Required public data missing", len(public_files))
print("PUBLIC DATASET VERIFIED:", f"https://huggingface.co/datasets/{DATASET_REPO_ID}")


## 4. Add published repositories to the required existing public collections

In [ ]:
for repo_id, _, _ in ADAPTERS.values():
    api.add_collection_item(
        collection_slug=MODEL_COLLECTION_SLUG,
        item_id=repo_id,
        item_type="model",
        exists_ok=True,
    )
api.add_collection_item(
    collection_slug=DATA_COLLECTION_SLUG,
    item_id=DATASET_REPO_ID,
    item_type="dataset",
    exists_ok=True,
)
for collection_slug in (MODEL_COLLECTION_SLUG, DATA_COLLECTION_SLUG):
    collection = api.get_collection(collection_slug)
    print("PUBLIC COLLECTION:", f"https://huggingface.co/collections/{collection_slug}")
    print("Items:", [(item.item_type, item.item_id) for item in collection.items])

print("SUCCESS: Original SW12/SW17 weights and exact matched training data are publicly uploaded.")
print("This notebook does not retrain, relabel, or modify the model weights.")
print("GitHub: https://github.com/githubshaurya/ShadowWork-SHIFT")
